# Multi-Objective Contextual Bandit with Amazon SageMaker AI

This notebook is a hands-on walkthrough of the multi-objective **LinUCB** contextual
bandit that accompanies the AWS blog post. You'll build intuition for how the bandit
chooses content, watch it learn on synthetic data, see how tuning the exploration
parameter `alpha` changes behavior, and review how the same model runs as a batch job
with Amazon SageMaker AI.

Everything here runs on **synthetic data** — no AWS account or proprietary data needed.

**Contents**
1. The problem: which content do you show each visitor?
2. LinUCB intuition (estimate + uncertainty bonus)
3. Optimizing the whole funnel (multi-objective)
4. Watch the bandit learn
5. Tuning the exploration parameter `alpha`
6. From notebook to production: the SageMaker AI batch pattern

## Setup

We add the repo's `src/` and `examples/` folders to the path so we can import the
bandit and the synthetic environment. Only NumPy is required; plots are shown if
matplotlib is installed, otherwise we print a text summary.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('../examples'))

import numpy as np
from linucb import LinUCBDisjoint, MultiObjectiveLinUCB
from synthetic_data import SyntheticFunnelEnv

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except ImportError:
    HAVE_MPL = False
    print('matplotlib not installed - will print text summaries instead of plots')

## 1. The problem: which content do you show each visitor?

When generative AI can produce hundreds of on-brand content variations, the hard part
is *selection*: which variation should each visitor see? A **multi-armed bandit** treats
each variation as an **arm**, tries arms against live traffic, and shifts traffic toward
the ones that perform — while still exploring the rest.

A **contextual** bandit goes further: it conditions the choice on a **context vector** of
signals about the visit (behavioral features), so it can pick a different best arm for
different visitors. That is personalization.

## 2. LinUCB intuition: estimate + uncertainty bonus

LinUCB assumes each arm's expected reward is a linear function of the context. For each
arm it keeps two running tallies, updated on every impression:

- `A` — the *experience ledger* (`A += x·xᵀ`, starts at the identity matrix)
- `b` — the *reward ledger* (`b += reward · x`)

The score for an arm given context `x` is:

```
score(a, x) = θᵀ·x  +  alpha · sqrt(xᵀ · A⁻¹ · x)
              estimate        uncertainty bonus       (θ = A⁻¹·b)
```

The first term is the predicted reward; the second is an optimism bonus that is **large**
where the arm has seen few similar visitors and **shrinks** as it gains experience. That
is how exploration and exploitation are balanced automatically.

In [ ]:
# A tiny single-objective bandit: 3 arms, 4 features.
bandit = LinUCBDisjoint(n_arms=3, n_features=4, alpha=1.0)
context = np.array([1.0, 0.5, -0.2, 0.1])

print('Initial UCB scores (all equal - no experience yet):')
print(np.round(bandit.ucb_scores(context), 4))

# Give arm 1 a few positive rewards for this kind of context.
for _ in range(5):
    bandit.update(arm=1, context=context, reward=1.0)

print('\nAfter rewarding arm 1 on this context:')
print(np.round(bandit.ucb_scores(context), 4))
print('Selected arm:', bandit.select_arm(context))

Notice arm 1's score rose (its estimate went up) while its uncertainty bonus for this
context shrank as it accumulated experience. The other arms are unchanged.

## 3. Optimizing the whole funnel (multi-objective)

Our reward is not a single click. We care about a visitor who **starts** an application,
**submits** it, and gets **approved**. These stages don't move together: content that
wins lots of starts can attract a crowd that rarely gets approved. Optimizing a single
stage causes the **seesaw problem** — one metric goes up while another slides down.

`MultiObjectiveLinUCB` runs **one LinUCB per stage** and selects the arm that maximizes a
**weighted sum** of the three UCB scores. The weights encode how much each stage matters
and sum to 1.

In [ ]:
model = MultiObjectiveLinUCB(
    n_arms=5, n_features=4, alpha=1.0,
    weights=(0.33, 0.33, 0.34),  # start, submit, approve
)
context = np.array([0.4, -0.1, 0.8, 0.2])

# One observed outcome for arm 2: started + submitted, not yet approved.
model.update(arm=2, context=context, r_start=1, r_submit=1, r_approve=0)

print('Combined (weighted) UCB scores per arm:')
print(np.round(model.combined_scores(context), 4))
print('Selected arm:', model.select_arm(context))

## 4. Watch the bandit learn

We stream synthetic visitors through the bandit, updating it online, and compare its
average funnel reward against a random-arm baseline. The synthetic environment has a
hidden 'best arm' per context, so a learner should pull ahead of random over time.

In [ ]:
N_ARMS, N_FEATURES, N_ROUNDS = 20, 8, 4000
WEIGHTS = (0.33, 0.33, 0.34)

env = SyntheticFunnelEnv(N_ARMS, N_FEATURES, seed=7)
model = MultiObjectiveLinUCB(N_ARMS, N_FEATURES, alpha=1.0, weights=WEIGHTS)
rng = np.random.default_rng(7)

def reward(o):
    return WEIGHTS[0]*o[0] + WEIGHTS[1]*o[1] + WEIGHTS[2]*o[2]

bandit_r, random_r = [], []
for t in range(N_ROUNDS):
    ctx = env.sample_context()
    arm = model.select_arm(ctx)
    o = env.step(arm, ctx)
    model.update(arm, ctx, *o)
    bandit_r.append(reward(o))
    random_r.append(reward(env.step(int(rng.integers(0, N_ARMS)), ctx)))

def moving_avg(x, w=200):
    x = np.asarray(x, float)
    return np.convolve(x, np.ones(w)/w, mode='valid')

b_ma, r_ma = moving_avg(bandit_r), moving_avg(random_r)
print(f'Bandit avg reward (last 25%): {np.mean(bandit_r[-1000:]):.4f}')
print(f'Random avg reward (last 25%): {np.mean(random_r[-1000:]):.4f}')

if HAVE_MPL:
    plt.figure(figsize=(8, 4))
    plt.plot(b_ma, label='multi-objective LinUCB')
    plt.plot(r_ma, label='random baseline')
    plt.xlabel('round'); plt.ylabel('avg funnel reward (moving avg)')
    plt.title('The bandit learns and pulls ahead of random')
    plt.legend(); plt.tight_layout(); plt.show()
else:
    print('\nReward by quartile (bandit):',
          [round(float(np.mean(c)), 4) for c in np.array_split(bandit_r, 4)])

## 5. Tuning the exploration parameter `alpha`

`alpha` controls how much weight the model puts on the uncertainty bonus. Higher values
explore under-tested arms more aggressively; lower values exploit what's already known.
`alpha = 1.0` is a sensible default. Below we sweep a few values on the same environment.
In practice, tune `alpha` on warm-start data via offline replay, and keep it in a modest
range (roughly 0.1-2.0).

In [ ]:
def run(alpha, rounds=3000, seed=7):
    env = SyntheticFunnelEnv(N_ARMS, N_FEATURES, seed=seed)
    m = MultiObjectiveLinUCB(N_ARMS, N_FEATURES, alpha=alpha, weights=WEIGHTS)
    total = 0.0
    for _ in range(rounds):
        ctx = env.sample_context()
        a = m.select_arm(ctx)
        o = env.step(a, ctx)
        m.update(a, ctx, *o)
        total += reward(o)
    return total / rounds

for alpha in [0.0, 0.5, 1.0, 2.0]:
    print(f'alpha={alpha:>4}:  avg reward = {run(alpha):.4f}')

## 6. From notebook to production: the SageMaker AI batch pattern

In production the bandit runs on a periodic (for example, weekly) cadence, because
approval feedback is delayed by days. The pattern is:

1. A scheduled **Amazon SageMaker AI Processing** job loads the latest model state from
   Amazon S3.
2. It updates the model on recent feedback and scores new prospects to pick an arm each.
3. It writes the updated model state back to S3 (under a new dated prefix, giving a
   versioned history and easy rollback) and publishes per-entity recommendations to a
   low-latency store that the live page reads with a single lookup.

See `src/batch_job.py` for the entry point and `deploy/run_processing_job.py` for the
launcher. The model serializes to plain NumPy arrays (no pickle), shown below.

In [ ]:
# Model state is a flat dict of NumPy arrays - safe to store and reload.
state = model.get_state()
print('State keys:', list(state.keys()))

# Round-trip: rebuild an identical model from its state (what S3 load/save does).
restored = MultiObjectiveLinUCB.from_state(state)
ctx = env.sample_context()
print('Same arm chosen after reload:',
      restored.select_arm(ctx) == model.select_arm(ctx))

## Next steps

- Swap the synthetic environment for your own arms (content variations) and context
  features.
- Adjust the objective `weights` to reflect your business priorities.
- Run `deploy/run_processing_job.py` to execute the batch step on Amazon SageMaker AI,
  and schedule it (SageMaker AI Pipelines or Amazon EventBridge) for a weekly cadence.

There are no AWS resources created by this notebook, so there is nothing to clean up.